# **NHS Talking Therapies 2019–2025: National Trends**


### **Table of Contents**

1. Service Demand & Treatment Funnel Overview

2. Recovery & Outcomes Trends

3. Waiting Time & Appointments Efficiency

###**Summary:**

All three dimensions reveal a consistent pattern: While the demand for NHS Talking Therapies has grown substantially in the post-pandemic period, the system's ability to retain patients through to completed treatment has not kept pace:

* Treatment funnel proportions remained consistent.
* Recovery rates remain below pre-pandemic levels.
* Fewer than 4 in 10 referrals complete treatment.
* Waiting times between appointments are rising.



###  **Notes: Data structure and quality**

**1. Analytical Timeframe: Year**

Data is collected by NHS financial year, covering from **April to March**. For example, 2020/21 refers to the period from April 2020 to March 2021.

**2. Analytical Scope: OrgType**

Due to NHS structural reforms, national trends will be analysed using England-level data across all six years (2019/20–2024/25), while regional breakdowns will use ICB-level data covering the most recent three years (2022/23–2024/25).


**3. Variable Naming**

As the variable naming system occasionally changes across 2019–2025, I will clean and standardise variable names when exploring each section.
(e.g. `Accessing Services` vs `First Treatment`, `Mean wait` vs `Mean Wait`)

**4. Waiting Time Breakdown**

Waiting time data is further broken down by VariableA (e.g. "Accessing Services", "First to Second Treatment"), which represents different stages of the referral pathway. This distinction is relevant to the interpretation of Section 3.1.

### **Column Reference**

|Index|Non-Null Count|Null Count|Dtype|Metadata |
|---|---|---|---|---|
|Year|483244|0|object|Financial year \(e\.g\. 2019/20\)|
|OrgType|483244|0|object|Organisation type \(England / ICB / SubICB / Provider\)|
|OrgCode|483244|0|object|ODS organisation code|
|OrgName|483244|0|object|ODS organisation name|
|VariableType|483244|0|object|Category type \(e\.g\. Age Group, Gender, Waiting Time\)|
|VariableA|479354|3890|object|High-level category value|
|VariableB|262693|220551|object|Sub-category value \(NaN if not applicable\)|
|Count\_ReferralsReceived|285279|197965|float64|Referrals received in the year|
|Count\_AccessingServices|297637|185607|float64|Referrals with a first attended treatment appointment|
|Count\_FinishedCourseTreatment|283140|200104|float64|Referrals completing ≥2 treatment appointments|
|Percentage\_FinishedCourseTreatment|277170|206074|float64|% finishing treatment \(interpretation varies by VariableType\)|
|Count\_EndedReferrals|283883|199361|float64|Total referrals ended in the year|
|Count\_EndedBeforeTreatment|227580|255664|float64|Referrals ended with no treatment appointments|
|Mean\_ApptsFinishedCourseTreatment|254928|228316|float64|Mean number of appointments for completed courses|
|Mean\_WaitAccessingServices|3859|479385|float64|Mean days from referral to 1st &  1st to 2nd treatment |
|Mean\_WaitFinishedCourseTreatment|4482|478762|float64|Same as above but for completed courses only|
|Percentage\_Improvement|232007|251237|float64|% showing reliable improvement|
|Percentage\_NoReliableChange|199140|284104|float64|% showing no reliable change|
|Percentage\_Deterioration|138057|345187|float64|% showing reliable deterioration|
|Percentage\_Recovery|216902|266342|float64|% moved to recovery \(excludes those not at caseness at intake\)|
|Percentage\_ReliableRecovery|214609|268635|float64|% showing both recovery and reliable improvement|




In [3]:
# @title Data Loading { display-mode: "form" }

import duckdb
import pandas as pd
import numpy as np

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/NHS_data_analysis_project/NHS_Talking_Therapies_Cleaned_All_Years.csv")
df.head()

Mounted at /content/drive


,Year,OrgType,OrgCode,OrgName,VariableType,VariableA,VariableB,Count_ReferralsReceived,Count_AccessingServices,Count_FinishedCourseTreatment,...,Percentage_Improvement,Percentage_NoReliableChange,Percentage_Deterioration,Percentage_Recovery,Percentage_ReliableRecovery,Mean_TreatmentAppointmentsRecovery,Mean_TreatmentAppointmentsImprovement,Mean_TreatmentAppointmentsNoChange,Mean_TreatmentAppointmentsDeterioration,Mean_TreatmentAppointmentsReliableRecovery
0,2019/20,England,All,All,Total,NaN,NaN,1694790.0,1165653.0,606192.0,...,67.0,26.2,5.8,51.1,48.5,7.7,7.6,5.6,6.0,7.8
1,2019/20,England,All,All,Age Group,Under 16,NaN,2359.0,1167.0,918.0,...,65.1,29.6,4.9,59.3,53.8,NaN,NaN,NaN,NaN,NaN
2,2019/20,England,All,All,Age Group,16 to 17,NaN,32693.0,20094.0,6601.0,...,59.2,32.6,7.3,45.0,41.3,NaN,NaN,NaN,NaN,NaN
3,2019/20,England,All,All,Age Group,18 to 35,NaN,860706.0,557942.0,285222.0,...,65.9,27.2,6.0,48.1,45.5,NaN,NaN,NaN,NaN,NaN
4,2019/20,England,All,All,Age Group,36 to 64,NaN,693594.0,506136.0,272481.0,...,67.9,25.4,5.7,52.6,50.0,NaN,NaN,NaN,NaN,NaN


## **1. Service Demand & Treatment Funnel Overview**


* Referrals declined in 2020/21 before recovering in 2021/22 and exceeding pre-pandemic levels (2019/20) by **117,806**, reflecting both the disruption caused by COVID-19 and the following surge in demand for mental health support.
* On average across the six-year period, 31.2% of referrals did not access treatment, and 30.8% accessed but did not finish treatment, which means fewer than 4 in 10 referrals had finished treatment. Despite overall fluctuations, including the COVID-19 period, the proportions remained broadly consistent over time.
* Notably, the number of patients who finished treatment remained stable throughout the COVID-19 period, despite the overall decline in referral volumes.

In [4]:
# @title Double click to show code { display-mode: "form" }

# Filter data
new_df = df[ (df['OrgType']=='England') & (df['VariableType']=='Total')]

df_treatment = new_df[[
  'Year','OrgName','VariableType',
  'Count_ReferralsReceived',
  'Count_AccessingServices',
  'Count_FinishedCourseTreatment']]

df_treatment = df_treatment.rename(columns={
    'Count_ReferralsReceived': 'Referrals Received',
    'Count_AccessingServices': 'Accessing Services',
    'Count_FinishedCourseTreatment': 'Finished Course Treatment'
})

dif = df_treatment[df_treatment['Year'] == '2021/22']['Referrals Received'].values[0] - df_treatment.loc[df_treatment['Year'] == '2019/20']['Referrals Received'].values[0]
print(f"Referrals difference (2021/22 vs 2019/20): {dif}")

# Service Funnel Conversion Rates

conversion_df = pd.DataFrame({
    'Year': df_treatment['Year'],
    'Referral to Accessing (%)': df_treatment['Accessing Services'] / df_treatment['Referrals Received'] * 100,
    'Lost at Accessing (%)': 100 - df_treatment['Accessing Services'] / df_treatment['Referrals Received'] * 100,
    'Accessing to Finished (%)': df_treatment['Finished Course Treatment'] / df_treatment['Referrals Received'] * 100,
    'Lost at Finishing (%)': (df_treatment['Accessing Services'] - df_treatment['Finished Course Treatment']) / df_treatment['Referrals Received'] * 100
}).set_index('Year')

print(f"Referrals Received --> Accessing Services: {conversion_df['Lost at Accessing (%)'].mean().round(1)}% loss.")
print(f"Accessing Services --> Finished Course Treatment: {conversion_df['Lost at Finishing (%)'].mean().round(1)}% loss.")

# Visualize data
fig = px.line(
    df_treatment, x="Year", y=["Referrals Received",'Accessing Services','Finished Course Treatment'],
    markers=True,
    title='Total Referrals Received 2019-2025',
    labels={
    'value': 'Count',
    'variable': 'Metric',
    'Year': 'Year'
    })

fig.update_layout(width=1000,height=500)
fig.show()



Referrals difference (2021/22 vs 2019/20): 117806.0
Referrals Received --> Accessing Services: 31.2% loss.
Accessing Services --> Finished Course Treatment: 30.8% loss.


## **2. Recovery & Outcomes Trends**

## 2.1 Recovery vs Reliable Recovery
* The percentage of recovery vs reliable recovery gap across six years remains broadly stable. However, both metrics fluctuated notably: Rates rose from 2019/20 to 2020/21, subsequently declined to their lowest in 2022/23, then rose steadily until 2024/25, despite never recovering to pre-pandemic levels.
* Given the referrals volume shrank in 2020/21, the higher recovery rates appear counterintuitive and would require further investigation.

*Recovery: Recovery_Flag = True*

*Reliable Recovery: Recovery_Flag = True & ReliableImprovement_Flag = True*

In [5]:
# @title Double click to show code { display-mode: "form" }
df_recovery = new_df[[
  'Year','OrgName','VariableType',
  'Percentage_Recovery',
  'Percentage_ReliableRecovery']]

df_recovery = df_recovery.rename(columns={
    'Percentage_Recovery': 'Percentage of Recovery',
    'Percentage_ReliableRecovery': 'Percentage of Reliable Recovery'
})

fig = px.line(
    df_recovery, x="Year", y=['Percentage of Recovery','Percentage of Reliable Recovery'],
    markers=True,
    title='Recovery Trend 2019-2025',
    labels={
    'value': 'Percentage',
    'variable': 'Metric',
    'Year': 'Year'
    })

fig.update_layout(width=1000,height=500)
fig.update_yaxes(ticksuffix="%")

fig.show()

## 2.2 Improvement vs Deterioration

* The percentage of improvement vs deterioration recovery gap across six years remains broadly stable.

In [6]:
# @title Double click to show code { display-mode: "form" }

new_df = df[ (df['OrgType']=='England') & (df['VariableType']=='Total')]
df_improv = new_df[[
  'Year','OrgName','VariableType',
  'Percentage_Improvement',
  'Percentage_Deterioration']]

df_improv = df_improv.rename(columns={
    'Percentage_Improvement': 'Percentage of Improvement',
    'Percentage_Deterioration': 'Percentage of Deterioration'
})

fig = px.line(
    df_improv, x="Year", y=['Percentage of Improvement','Percentage of Deterioration'],
    markers=True,
    title='Improvement vs Deterioration Trends 2019-2025',
    labels={
    'value': 'Percentage',
    'variable': 'Metric',
    'Year': 'Year'
    })

fig.update_layout(width=1000,height=500)
fig.update_yaxes(ticksuffix="%")
fig.show()

## 2.3 Distribution Count: Total Referrals Ended vs Referrals Ended Before Treatment Began

* All three metrics—Count Ended Referrals, Count Ended Before Treatment, and the percentage ended before treatment—reached their lowest point in 2020/21.
* And despite the post-pandemic increase, the percentage ended before treatment has remained below pre-pandemic level.

In [7]:
# @title Double click to show code { display-mode: "form" }

new_df = df[ (df['OrgType']=='England') & (df['VariableType']=='Total')]
df_endtreatment = new_df[[
  'Year','OrgName','VariableType',
  'Count_EndedBeforeTreatment',
  'Count_EndedReferrals']]

df_endtreatment = df_endtreatment.rename(columns={
    'Count_EndedBeforeTreatment': 'Count Ended Before Treatment',
    'Count_EndedReferrals': 'Count Ended Referrals'
})

df_endtreatment['% Ended Before Treatment'] = df_endtreatment['Count Ended Before Treatment'] / df_endtreatment['Count Ended Referrals'] * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Total Referrals Ended vs Referrals Ended Before Treatment',
            'Percentage of Referrals Ended Before Treatment')
)

fig1 = px.line(
    df_endtreatment, x="Year", y=['Count Ended Referrals','Count Ended Before Treatment'],
    markers=True,
    labels={
    'value': 'Count',
    'variable': 'Metric',
    'Year': 'Year'
    })
fig2 = px.line(
    df_endtreatment, x="Year", y=['% Ended Before Treatment'],
    markers=True,
    color_discrete_sequence=['#2ca02c'],
    labels={
    'value': 'Percentage',
    'variable': 'Metric',
    'Year': 'Year'
    })

for trace in fig1.data:
  fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
  fig.add_trace(trace, row=1, col=2)

fig.update_yaxes(ticksuffix='%', row=1, col=2)
fig.show()


## **3. Recovery & Outcomes Trends**

## 3.1 Waiting Time
* Notably, the wait time between the first and second treatment appointment is substantially longer than the wait to access services.
* The wait time between the first and second treatment has risen steadily since 2021/22, reaching over 65 days by 2024/25. This suggests the system struggles more with getting patients into the actual treatment process(from access to treatment) than with initial access.
* Patients who completed treatment waited considerably longer for their second appointment than the average referral accessing services.

*Mean_WaitAccessingServices:  Average wait time for referrals who had first accessed services(First Treatment) in the financial year.*

*Mean_WaitCompletedTreatment:	 Average wait time for referrals who had finished a course of treatment in the financial year.*

In [8]:
# @title Double click to show code { display-mode: "form" }

new_df = df[ (df['OrgType']=='England') & (df['VariableType']=='Waiting Time')]

df_wait = new_df[new_df['Mean_WaitAccessingServices'].notna()]
df_wait = df_wait[['Year','OrgType','VariableA','OrgName','Mean_WaitAccessingServices','Mean_WaitFinishedCourseTreatment']]

df_wait = df_wait.rename(columns={
    'Mean_WaitAccessingServices': 'Mean Wait to Access Services (days)',
    'Mean_WaitFinishedCourseTreatment': 'Mean Wait - Finished Course of Treatment (days)'
    })
df_wait['VariableA'] = df_wait['VariableA'].replace({
    'First treatment': 'Accessing Services',
    'Accessing services': 'Accessing Services',
    'First to Second Treatment': 'First to Second Treatment',
    'First to Second treatment': 'First to Second Treatment'
})
plot1 = df_wait[df_wait['VariableA']=='Accessing Services']
plot2 = df_wait[df_wait['VariableA']=='First to Second Treatment']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Accessing Services', 'First to Second Treatment')
)

fig1 = px.line(
    plot1, x='Year',
    y=['Mean Wait to Access Services (days)', 'Mean Wait - Finished Course of Treatment (days)'],
    markers=True,
    labels={'value': 'Days','variable': 'Metric','Year': 'Year'})

fig2 = px.line(
    plot2, x='Year',
    y=['Mean Wait to Access Services (days)', 'Mean Wait - Finished Course of Treatment (days)'],
    markers=True,
    labels={'value': 'Days','variable': 'Metric','Year': 'Year'})

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Accessing Services', 'First to Second Treatment'))

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=2)

fig.update_yaxes(ticksuffix=" days")
fig.show()


## 3.2 The Number of Appointments vs Outcome

* Patients who gained positive outcomes (Recovery, Improvement, Reliable Recovery) tend to attend more appointments than those with more negative outcomes (No Change, Deterioration).
* However, this does not necessarily indicate that more appointments mean better outcomes, as more engaged patients are likely both attend more sessions and recover.


In [9]:
# @title Double click to show code { display-mode: "form" }

new_df = df[(df['OrgType']=='England') & (df['VariableType']=='Total')]

cols = ['Mean_TreatmentAppointmentsRecovery', 'Mean_TreatmentAppointmentsImprovement',
        'Mean_TreatmentAppointmentsNoChange', 'Mean_TreatmentAppointmentsDeterioration',
        'Mean_TreatmentAppointmentsReliableRecovery']

df_appoint = new_df[['Year', 'OrgType', 'VariableType'] + cols].dropna(subset=cols)

df_appoint = df_appoint.rename(columns={
    'Mean_TreatmentAppointmentsRecovery': 'Recovery',
    'Mean_TreatmentAppointmentsImprovement': 'Improvement',
    'Mean_TreatmentAppointmentsNoChange': 'No Change',
    'Mean_TreatmentAppointmentsDeterioration': 'Deterioration',
    'Mean_TreatmentAppointmentsReliableRecovery': 'Reliable Recovery'
})

fig = px.line(
    df_appoint, x="Year",
    y=['Recovery', 'Improvement', 'No Change', 'Deterioration', 'Reliable Recovery'],
    markers=True,
    title='Average Treatment Appointments by Outcome 2019-2025',
    labels={'value': 'Mean Appointments', 'variable': 'Outcome', 'Year': 'Year'}
)

fig.update_layout(width=1000, height=500)
fig.show()